# <center> Responsibe Banking Assistant </center>


<font color='red'>**Important**: </font>
- **If the kernel doesn't attach automatically, or if you need to restart the kernel, please choose 'conda_python3' kernel.**
- Run each block of code one by one (starting from the top) either by clicking on "Run" on the menu above or by pressing SHIFT+ENTER on your keyboard.
- For the blocks that you need to complete, you see **"YOUR TURN"** in the title.
- Don't alter the code in places without ``YOUR TURN`` in the title. Just run them. 
- Have fun!

## Introduction

***

Welcome to this challenge! 

You are working as a data scientist for Octank Bank, a leading financial institution. As online banking services are rapidly expanding, the bank aims to introduce a digital assistant to assist customers with account opening procedures, provide information about the bank, current offers, and other relevant details. However, the bank is cautious about potential misuse of the digital assistant and wants to avoid providing investment advice.

In this challenge, you will first learn how to set up an Amazon Bedrock Knowledge Base. Next, you will learn how to retrieve data from the Amazon Bedrock Knowledge Bases. After that, you will learn how to configure Amazon Bedrock Guardrails to ensure safe interactions between customers and your digital assistant. This involves implementing safeguards customized to your use cases and responsible AI policies.
 

***

## Overview
Here is what you are going to do in this challenge:


   *[Task 1 : Knowledge Base](#test-knowledge-base)
   
   *[Task 2 : Guardrails](#test-the-guardrails)
   
   *[Task 3 : Mask PII Data](#mask-pii-data)


<a id='Set-Up'> </a>

### Set Up

<font color='red'> Important: Please make sure to enable `Titan Embeddings G1 - Text` and `Nova Lite` model access in Amazon Bedrock Console, as the notebook will use these models for completing the task.</font>

#### Follow the below steps

#### 1. Go to AWS console
#### 2. Search Amazon Bedrock in the search bar
#### 3. On the left hand side menu Bedrock configuration, Click on Model access
#### 4. Click on the Enable specific models
<img style="text-align:left"/>
  <img src="https://aws-jam-challenge-resources.s3.amazonaws.com/responsible-banking-assistant/1_Model_Access_Enable.png" alt="drawing" width="1000" />

#### 5. Select the model mentioned above
<img style="text-align:left"/>
  <img src="https://aws-jam-challenge-resources.s3.amazonaws.com/responsible-banking-assistant/model_access.png" alt="drawing" width="1000" />
  
#### 6. Click on Next
#### 7. Click on submit
<img style="text-align:left"/>
  <img src="https://aws-jam-challenge-resources.s3.amazonaws.com/responsible-banking-assistant/model_confirmation.png" alt="drawing" width="1000" />


***

Before executing the notebook, there are some initial steps required for set up. 

***

In [ ]:
!pip install -qU boto3 awscli
%pip install -U opensearch-py==2.3.1

#### Permissions and environment variables

***
First we need to set up and authenticate the use of AWS services. Here, we use the execution role associated with the current notebook as the AWS account role with required permissions.

***

In [ ]:
import json
import time
import uuid
import boto3
import json
import os
from botocore.exceptions import ClientError
import pprint
from utility import create_bedrock_execution_role, create_oss_policy_attach_bedrock_execution_role, create_policies_in_oss, interactive_sleep
import random
import time

s3_client = boto3.client('s3')
iam_client = boto3.client('iam')
sts_client = boto3.client('sts')
session = boto3.session.Session()
unique_id = str(uuid.uuid4())[:4]
region = session.region_name
bedrock = boto3.client("bedrock",region_name=region)
account_id = sts_client.get_caller_identity()["Account"]


In [ ]:
import pprint
pp = pprint.PrettyPrinter(indent=2)

In [ ]:
suffix = random.randrange(200, 900)
sts_client = boto3.client('sts')
boto3_session = boto3.session.Session()
region_name = boto3_session.region_name
bedrock_agent_client = boto3_session.client('bedrock-agent', region_name=region_name)
service = 'aoss'
s3_client = boto3.client('s3')
account_id = sts_client.get_caller_identity()["Account"]
pp = pprint.PrettyPrinter(indent=2)

In [ ]:
# Fetches the Amazon S3 Data source associated with the Amazon Bedrock KnowledgeBase
s3_buckets = s3_client.list_buckets()
for bucket in s3_buckets['Buckets']:
    if 'bankingassistantbucket' in bucket['Name'].lower():
        bucket_name = bucket['Name']
        break
print(f'The S3 bucket associated with this challenge is: {bucket_name}')

### Define Helper Functions (please don't modify)

In [ ]:
# This is a helper function that validates your final answers for various tasks, PLEASE DONT MODIFY THIS
def submit_your_answer(task_file_name, task_response):
    '''Upload your response to S3 in json format. This acts as your submission!
       Arguments:
           task_file_name (str): name of the task file you are submitting 
           task_response (Dictionary): Response dictionary 
    ''' 
    task_json = json.dumps(task_response)
    with open(task_file_name, 'w') as f:
        f.write(task_json)
    s3_client.upload_file(Bucket=bucket_name, Filename=os.path.join('./', task_file_name), Key=task_file_name)


In [ ]:
# Fetch the knowledge base id from your AWS JAM account
get_knowledgebase_id=bedrock_agent_client.list_knowledge_bases()
kb_id = (get_knowledgebase_id['knowledgeBaseSummaries'][0]['knowledgeBaseId'])
kb_id

<font color='red'>**Important**: </font>
### Sync KB with the S3 datasource

### Please complete the below steps before move to next cell

#### 1. Go to AWS console
#### 2. Search for bedrock in the search bar
#### 3. Click on the hamburger icon on the upper left corner of to expand the left hand side menu 
#### 4. On the left hand side menu Under Builder tools, Click on Knowledged bases
#### 5. Click on the knowledge base link
#### 6. Scroll down to reach the section "Data source" 
#### 7. Select the radio button to the left of the data source [there is only one]
#### 8. Click the 'sync' button

<br>

<img style="text-align:left"/>
  <img src="https://aws-jam-challenge-resources.s3.amazonaws.com/responsible-banking-assistant/KB_Sync.png" alt="drawing" width="1000" />
</p>


In [ ]:
get_kb_response = bedrock_agent_client.get_knowledge_base(knowledgeBaseId = kb_id)

In [ ]:
# Updating Nova Lite LLM model in the pre-created Amazon KnowledgeBase
bedrock_agent_runtime_client = boto3.client("bedrock-agent-runtime", region_name=region_name)
nova_model_ids = [ ["Nova Lite", "amazon.nova-lite-v1:0"]]

In [ ]:
def ask_bedrock_llm_with_knowledge_base(query: str, model_arn: str, kb_id: str) -> str:
    response = bedrock_agent_runtime_client.retrieve_and_generate(
        input={
            'text': query
        },
        retrieveAndGenerateConfiguration={
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': model_arn
            }
        },
    )

    return response

<a id='test-knowledge-base'> </a>
### <font color='17C822'> YOUR TURN: Task 1 : Test Knowledge Base</font>
<b>Write a prompt to find the total number of bank branch available globally

In [ ]:
query = "WRITEYOURPROMPTHERE"

for model_id in nova_model_ids:
    model_arn = f'arn:aws:bedrock:{region_name}::foundation-model/{model_id[1]}'
    response = ask_bedrock_llm_with_knowledge_base(query, model_arn, kb_id)
    generated_text = response['output']['text']
    citations = response["citations"]
    contexts = []
    for citation in citations:
        retrievedReferences = citation["retrievedReferences"]
        for reference in retrievedReferences:
            contexts.append(reference["content"]["text"])
    print(f"---------- Generated using {model_id[0]}:")
    pp.pprint(generated_text )
    print(f'---------- The citations for the response generated by {model_id[0]}:')
    pp.pprint(contexts)
    print()

In [ ]:
# Update total number of bank branch you are getting after executing above cell
total_branch= ''

##################################################################################################################
### SUBMISSION (DON'T CHANGE THE CODE)
Run the following cell to submit your response. <font color='red'> **DON'T CHANGE THIS LINE** </font>. Just run it as it is once you have completed the ``total_branch`` dictionary above. This is your submission for Task #1. You'll receive credit shortly if you answer is correct.
##################################################################################################################

In [ ]:
# DON'T CHANGE THIS LINE OR YOUR SUBMISSION FAILS
submit_your_answer('task1.json', total_branch)

## Create the guardrails

For illustrating the use of guardarails with our virtual financial agent use case, we'll create a guardrail to simulate topics we don't want our chatbot to respond to.

   InvestmentAdvice - Any question or instruction related to financial information, transactions, or related.
   

We'll also enable the pre-defined content filters for toxic or harmful language.


In [ ]:
response = bedrock.create_guardrail(
    name="banking-assistant-guardrail-{}".format(unique_id),
    description="Only respond to the banking related questions, is protected against the most common prompt mis-use threads, and doesn't answer to competitor's references or financial advice.",
    topicPolicyConfig={
              'topicsConfig': [
                  {
                      'name': 'InvestmentAdvice',
                      'definition': "Statements or questions about finances, transactions or monetary advise.",
                      'examples': [
                          "What are the cheapest rates?",
                          "Where can I invest to get rich?",
                          "Can I get a guaranted return if I invest in mutual fund?"
                      ],
                      'type': 'DENY'
                  }

              ]
          },
    contentPolicyConfig={
              'filtersConfig': [
                  {
                      "type": "SEXUAL",
                      "inputStrength": "HIGH",
                      "outputStrength": "HIGH"
                  },
                  {
                      "type": "VIOLENCE",
                      "inputStrength": "HIGH",
                      "outputStrength": "HIGH"
                  },
                  {
                      "type": "HATE",
                      "inputStrength": "HIGH",
                      "outputStrength": "HIGH"
                  },
                  {
                      "type": "INSULTS",
                      "inputStrength": "HIGH",
                      "outputStrength": "HIGH"
                  },
                  {
                      "type": "MISCONDUCT",
                      "inputStrength": "HIGH",
                      "outputStrength": "HIGH"
                  },
                  {
                      "type": "PROMPT_ATTACK",
                      "inputStrength": "HIGH",
                      "outputStrength": "NONE"
                  }
              ]
          },
    wordPolicyConfig={
        'wordsConfig': [
            {
                'text': 'CompetitorX'
            },
            {
                'text': 'CompetitorY'
            }
        ],
        'managedWordListsConfig': [
            {
                'type': 'PROFANITY'
            }
        ]
    },
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': [
            {
                'type': 'NAME',
                'action': 'ANONYMIZE'
            },
            {
                'type': 'EMAIL',
                'action': 'ANONYMIZE'
            },
        ]
    },
    blockedInputMessaging="Sorry, your query violates our usage policies.",
    blockedOutputsMessaging="Sorry, I am unable to answer.",
)
guardrailId = response["guardrailId"]
print("The guardrail id is",response["guardrailId"])

In [ ]:
bedrock_runtime = boto3.client("bedrock-runtime", region_name=region)

In [ ]:
def call_bedrock_nova_model_with_guardrails(user_input):
    prompt = f"""You are an online banking assistant for OctankBank.


{user_input}"""
    
    input_body = {
        "messages": [
            {
                "role": "user", 
                "content": [{"text": prompt}]
            }
        ],
        "inferenceConfig": {
            "temperature": 0.7,
            "topP": 0.9,
            "maxTokens": 1000
        }
    }
    response = bedrock_runtime.invoke_model(
        modelId="amazon.nova-lite-v1:0",
        contentType="application/json",
        accept="application/json",
        body=json.dumps(input_body),
        trace="ENABLED",
        guardrailIdentifier= guardrailId,
        guardrailVersion= "DRAFT"
    )
    output_body = json.loads(response["body"].read().decode())
    action = output_body["amazon-bedrock-guardrailAction"]
    if action == "INTERVENED":
     print("Output text: {}".format(output_body["output"]["message"]["content"][0]["text"]))

<a id='test-the-guardrails'> </a>
### <font color='17C822'> YOUR TURN: Task 2 : Test guardrails</font>
<b>Write a prompt and ask the recommendation from assistant if you should invest in stocks

In [ ]:
call_bedrock_nova_model_with_guardrails("")


In [ ]:
#update guradrail_response 
guardrail_response=''

##################################################################################################################
### SUBMISSION (DON'T CHANGE THE CODE)
Run the following cell to submit your response. <font color='red'> **DON'T CHANGE THIS LINE** </font>. Just run it as it is once you have completed the ``guardrail_response`` dictionary above. This is your submission for Task #2. You'll receive credit shortly if you answer is correct.
##################################################################################################################

In [ ]:
# DON'T CHANGE THIS LINE OR YOUR SUBMISSION FAILS
submit_your_answer('task2.json', guardrail_response)

<a id='mask-pii-data'> </a>
### <font color='17C822'> YOUR TURN: Task 3 : Mask PII data</font>
You are a bank manager and you want to summarize the below account closing request transcript from customer 
where customer has requested Banking Assistant to close their existing account. 

<br>Below is the transcript

<br>Agent: Welcome to Octank bank. How can I help you today?
<br>Customer: I want to close my bank account.
<br>Agent: Sure, I can help you with the account closing. Can you please provide your booking ID?
<br>Customer: Yes, my account number is 650e8409.
<br>Agent: Thank you. Can I have your name and email for confirmation?
<br>Customer: My name is John Doe and my email is john.doe@test.com
<br>Agent: Thank you for confirming. I will go ahead and close your saving account.

<font color='red'>**Important**: </font>
<br>You need to add a regular expression pattern in the existing guardrail to mask the bank account number.

Provide below configuration of the regex pattern:
<B> 
<br>Name = account number
<br>Regex pattern = [0-9a-fA-F]{8}
<br>Guardrail behaviour = Mask

</B>


In [ ]:
chat_summary = "Please summarize the below account closing request transcript.Put the name, email and the account number to the top. Agent: Welcome to Octank bank. How can I help you today?Customer: I want to close my bank account.Agent: Sure, I can help you with the account closing. Can you please provide your account number?Customer: Yes, my account number is 650e8409.Agent: Thank you. Can I have your name and email for confirmation?Customer: My name is John Doe and my email is john.doe@test.com Agent: Thank you for confirming. I will go ahead and close your saving account."
call_bedrock_nova_model_with_guardrails(chat_summary)

In [ ]:
#update acount number based on above output text
account_number=''

### SUBMISSION (DON'T CHANGE THE CODE)
Run the following cell to submit your response. <font color='red'> **DON'T CHANGE THIS LINE** </font>. Just run it as it is once you have completed the ``account_number`` dictionary above. This is your submission for Task #4. You'll receive credit shortly if you answer is correct.
##################################################################################################################

In [ ]:
# DON'T CHANGE THIS LINE OR YOUR SUBMISSION FAILS
submit_your_answer('task4.json',  account_number)